# 44 · Chart's transmission tree, checked against networkx

## What this checks

**Chart**, set to **Transmission tree**, draws who infected whom as chains over time. This check covers release
0.25.2.

Before it draws, Chart works out three things for each case. Which chain it belongs to. Its generation, that is, how
many steps it is from the first case of its chain. And the row it sits on in the figure.

This check covers that work, not the drawing itself. A case put in the wrong chain or generation tells the wrong
story of how the outbreak spread.

## How it is checked

We rebuild the same chains with **networkx**, a Python library for networks of links. No code is shared with
Fieldwork.

networkx links each infector to the case they infected. A chain is every case linked, by any path, to the same first
case. That first case, the root, names no infector, or names one the table does not hold. A case's generation is one
more than the number of links between it and its root.

The data are the measles outbreak in Hagelloch, Germany, in 1861: 188 cases. Both sides read the same three columns.

## What should happen

1. **The same chains.** Every case is in the chain started by the same root, and the chains have the same sizes.
2. **The same generation** for every case.
3. **A valid order of rows.** Every case sits below its infector. The cases of each chain sit on rows next to each
   other.
4. **The same counts** of cases, chains, links, and cases that name no infector.

These criteria were fixed before the check was run.

In [ ]:
NEEDED = ["networkx"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the installed packages.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_44_transmission_tree import run

report = run(Path.cwd() / "fixtures" / "tree-hagelloch-001.json")
print("agrees:", report["agrees"])
c = report["counts"]
print(f"{c['cases']} cases in {c['chains']} chains, {c['links']} links; {c['noInfector']} name no infector")
l = report["largest"]
print(f"Largest chain: starts with case {l['root']}, {l['size']} cases over {l['generations']} generations")
print("Chain sizes:", report["chainSizes"])
print("Every case below its infector:", report["rowsBelowInfector"], "· each chain on consecutive rows:", report["chainsOnConsecutiveRows"])
print("Differs:", report["differs"] or "none")

## The result on record

First recorded on 10 November 2026, against release 0.12.0. Rerun on 26 November 2026, against release 0.25.2:
**Agrees**.

- **The counts.** 188 cases in 4 chains, of 173, 9, 5 and 1 cases. There are 184 links, and 4 cases name no
  infector.
- **Chains and generations.** Every case has the same chain and generation on both sides.
- **The largest chain.** It starts with case 184 and runs over 6 generations.
- **The rows.** Every case sits below its infector, and each chain is on rows next to each other.

The result agrees in CPython and in the browser kernel.

Running the cell above repeats the check on the same data.

## What this does not show

- Who infected whom. Both sides read the same 1861 reconstruction in the infector column.
- That the figure is drawn as computed. This checks the work the figure is drawn from, not the figure.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
e = report["environment"]
print("networkx:", e["networkx"])
print("runtime :", e["runtime"], "· Pyodide" if e["pyodide"] else "")